In [2]:
# # 02 — DA-WI Risk Relationships

# Frozen evidence layer for the synthetic-data pipeline.

# - DA-WI: India-specific risk-factor relationships (RRRs) and published weights.
# - NFHS-5: population-level prevalence/reference values where available in our CSV.
# - No synthetic generation or ML training is performed here.

# Source: Sabri et al. (2024), Table 3, DA-WI. The study used longitudinal data from 150 women in India; the weighted DA-WI reported AUC 0.803 for future severe IPV.

In [3]:
import pandas as pd
import numpy as np
import os

nfhs_violence=pd.read_csv('../data/processed/nfhs_violence_reference.csv')
nfhs_relevant=pd.read_csv('../data/processed/nfhs_relevant_indicators.csv')
print('NFHS violence reference:',nfhs_violence.shape)
print('NFHS relevant indicators:',nfhs_relevant.shape)

NFHS violence reference: (111, 4)
NFHS relevant indicators: (407, 9)


In [4]:
# ## 1. Published DA-WI factors

# The published RRR-to-weight rule is: <1.33→0, 1.33–1.79→1, 1.80–2.79→2, 2.80–3.79→3, ≥3.80→4. Strangulation was intentionally increased from 2 to 3.

In [6]:
# ============================================================
# DA-WI RISK FACTORS — PUBLISHED RELATIONSHIPS
# ============================================================

import pandas as pd

dawi_rows = [
    ("violence_escalation",
     "Increase in severity/frequency of physical violence",
     1.90, 2, "original_DA"),

    ("threat_to_kill",
     "He threatened to kill",
     1.76, 1, "original_DA"),

    ("violent_jealousy",
     "He is constantly/violently jealous",
     3.32, 3, "original_DA"),

    ("recent_separation",
     "She left him in the past year",
     2.32, 2, "original_DA"),

    ("lethal_weapon",
     "Used/threatened with a lethal weapon",
     2.64, 2, "original_DA"),

    ("avoids_arrest",
     "Lied about behavior/avoided being arrested",
     3.52, 3, "original_DA"),

    ("strangulation",
     "Choking/Strangulation",
     2.05, 3, "original_DA"),

    ("illegal_drug_use",
     "Partner uses illegal drugs",
     1.86, 2, "original_DA"),

    ("problem_drinking",
     "He is an alcoholic/problem drinker",
     1.87, 2, "original_DA"),

    ("violence_during_pregnancy",
     "She was beaten while pregnant",
     5.60, 4, "original_DA"),

    ("partner_capable_of_killing",
     "Partner is capable of killing her",
     2.04, 2, "original_DA"),

    ("suicide_threat_attempt",
     "She threatened/tried suicide",
     1.67, 1, "original_DA"),

    ("withholds_necessities",
     "He keeps necessities from her",
     10.32, 4, "additional_DA_WI"),

    ("intimidating_behavior",
     "He scares her with his behavior/body language",
     2.23, 2, "additional_DA_WI"),

    ("rumors",
     "He/his family spreads rumors about her",
     1.70, 1, "additional_DA_WI"),

    ("false_accusations",
     "He/his family makes false accusations",
     2.37, 2, "additional_DA_WI"),

    ("family_rejection",
     "Rejection by her family/community because of false accusations",
     2.41, 2, "additional_DA_WI"),

    ("social_isolation",
     "Husband isolates her from family/friends",
     2.69, 2, "additional_DA_WI"),

    ("inlaws_support_abuse",
     "In-laws support partner abuse",
     9.83, 4, "additional_DA_WI"),

    ("infertility_related_abuse",
     "Abuse for not being able to get pregnant",
     5.58, 4, "additional_DA_WI"),

    ("healthcare_neglect",
     "He/in-laws did not care for healthcare needs",
     4.12, 4, "additional_DA_WI"),

    ("family_honor_threat",
     "Threatened to harm/kill for family honor",
     2.83, 3, "additional_DA_WI"),

    ("leaving_threat",
     "Threatened to harm/kill her for leaving",
     2.11, 2, "additional_DA_WI"),

    ("hide_abuse",
     "Feels the need to hide husband abuse",
     2.47, 2, "additional_DA_WI"),

    ("lack_of_support",
     "Family/community members lack support",
     2.65, 2, "additional_DA_WI"),

    ("family_supports_abuse",
     "Parents/siblings support her abuse",
     2.81, 3, "additional_DA_WI")
]


# Convert to DataFrame
dawi = pd.DataFrame(
    dawi_rows,
    columns=[
        "feature",
        "dawi_item",
        "rrr",
        "weight",
        "source_group"
    ]
)

print("DA-WI factors:", len(dawi))

display(dawi)

DA-WI factors: 26


,feature,dawi_item,rrr,weight,source_group
0,violence_escalation,Increase in severity/frequency of physical vio...,1.90,2,original_DA
1,threat_to_kill,He threatened to kill,1.76,1,original_DA
2,violent_jealousy,He is constantly/violently jealous,3.32,3,original_DA
3,recent_separation,She left him in the past year,2.32,2,original_DA
4,lethal_weapon,Used/threatened with a lethal weapon,2.64,2,original_DA
5,avoids_arrest,Lied about behavior/avoided being arrested,3.52,3,original_DA
6,strangulation,Choking/Strangulation,2.05,3,original_DA
7,illegal_drug_use,Partner uses illegal drugs,1.86,2,original_DA
8,problem_drinking,He is an alcoholic/problem drinker,1.87,2,original_DA
9,violence_during_pregnancy,She was beaten while pregnant,5.60,4,original_DA


In [7]:
dawi=pd.DataFrame(dawi_rows,columns=['feature','dawi_item','rrr','weight','source_group'])
print('DA-WI factors:',len(dawi))
display(dawi)

DA-WI factors: 26


,feature,dawi_item,rrr,weight,source_group
0,violence_escalation,Increase in severity/frequency of physical vio...,1.90,2,original_DA
1,threat_to_kill,He threatened to kill,1.76,1,original_DA
2,violent_jealousy,He is constantly/violently jealous,3.32,3,original_DA
3,recent_separation,She left him in the past year,2.32,2,original_DA
4,lethal_weapon,Used/threatened with a lethal weapon,2.64,2,original_DA
5,avoids_arrest,Lied about behavior/avoided being arrested,3.52,3,original_DA
6,strangulation,Choking/Strangulation,2.05,3,original_DA
7,illegal_drug_use,Partner uses illegal drugs,1.86,2,original_DA
8,problem_drinking,He is an alcoholic/problem drinker,1.87,2,original_DA
9,violence_during_pregnancy,She was beaten while pregnant,5.60,4,original_DA


In [10]:
def rrr_to_weight(rrr):
    if rrr < 1.33:
        return 0
    elif rrr <= 1.79:
        return 1
    elif rrr <= 2.79:
        return 2
    elif rrr <= 3.79:
        return 3
    else:
        return 4


dawi["rule_weight"] = dawi["rrr"].apply(rrr_to_weight)

# Identify differences between generic RRR rule and published weight
dawi["weight_matches_rule"] = (
    dawi["rule_weight"] == dawi["weight"]
)

display(
    dawi[
        [
            "feature",
            "rrr",
            "rule_weight",
            "weight",
            "weight_matches_rule"
        ]
    ]
)

print("Mismatches:")
display(
    dawi[~dawi["weight_matches_rule"]]
)

print(
    "Expected mismatches:",
    (~dawi["weight_matches_rule"]).sum()
)

,feature,rrr,rule_weight,weight,weight_matches_rule
0,violence_escalation,1.90,2,2,True
1,threat_to_kill,1.76,1,1,True
2,violent_jealousy,3.32,3,3,True
3,recent_separation,2.32,2,2,True
4,lethal_weapon,2.64,2,2,True
5,avoids_arrest,3.52,3,3,True
6,strangulation,2.05,2,3,False
7,illegal_drug_use,1.86,2,2,True
8,problem_drinking,1.87,2,2,True
9,violence_during_pregnancy,5.60,4,4,True


Mismatches:


,feature,dawi_item,rrr,weight,source_group,rule_weight,weight_matches_rule
6,strangulation,Choking/Strangulation,2.05,3,original_DA,2,False


Expected mismatches: 1


In [11]:
expected_override = ["strangulation"]

unexpected_mismatches = dawi[
    (~dawi["weight_matches_rule"]) &
    (~dawi["feature"].isin(expected_override))
]

print("Expected DA-WI overrides:", expected_override)
print("Unexpected mismatches:", len(unexpected_mismatches))

if len(unexpected_mismatches) == 0:
    print("PASS: All DA-WI weights are consistent with the published rule and documented override.")
else:
    print("CHECK REQUIRED:")
    display(unexpected_mismatches)

Expected DA-WI overrides: ['strangulation']
Unexpected mismatches: 0
PASS: All DA-WI weights are consistent with the published rule and documented override.


In [ ]:
# 2. NFHS-5 → DA-WI mapping
# The downloaded NFHS CSV is aggregated by state/UT. We only mark a factor as directly supported when the file actually contains that concept. We do not invent missing NFHS measurements.

In [13]:
# ============================================================
# NFHS-5 → DA-WI MAPPING
# ============================================================

mapping_rows = [
    (
        "violence_escalation",
        "related",
        "Spousal violence is available; escalation itself is not directly measured."
    ),
    (
        "threat_to_kill",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "violent_jealousy",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "recent_separation",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "lethal_weapon",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "avoids_arrest",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "strangulation",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "illegal_drug_use",
        "not_available",
        "No direct partner drug-use indicator in this aggregated NFHS-5 CSV."
    ),
    (
        "problem_drinking",
        "context_only",
        "Alcohol prevalence is available, but partner problem drinking is not directly measured."
    ),
    (
        "violence_during_pregnancy",
        "direct_reference",
        "Physical violence during any pregnancy is directly reported."
    ),
    (
        "partner_capable_of_killing",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "suicide_threat_attempt",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "withholds_necessities",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "intimidating_behavior",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "rumors",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "false_accusations",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "family_rejection",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "social_isolation",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "inlaws_support_abuse",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "infertility_related_abuse",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "healthcare_neglect",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "family_honor_threat",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "leaving_threat",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "hide_abuse",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "lack_of_support",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    ),
    (
        "family_supports_abuse",
        "not_available",
        "No direct indicator in this NFHS-5 CSV."
    )
]

mapping = pd.DataFrame(
    mapping_rows,
    columns=[
        "feature",
        "nfhs_status",
        "mapping_note"
    ]
)

dawi_mapping = dawi.merge(
    mapping,
    on="feature",
    how="left"
)

print("Mapping rows:", len(mapping))
print("DA-WI mapping rows:", len(dawi_mapping))

display(dawi_mapping)

Mapping rows: 26
DA-WI mapping rows: 26


,feature,dawi_item,rrr,weight,source_group,rule_weight,weight_matches_rule,nfhs_status,mapping_note
0,violence_escalation,Increase in severity/frequency of physical vio...,1.90,2,original_DA,2,True,related,Spousal violence is available; escalation itse...
1,threat_to_kill,He threatened to kill,1.76,1,original_DA,1,True,not_available,No direct indicator in this NFHS-5 CSV.
2,violent_jealousy,He is constantly/violently jealous,3.32,3,original_DA,3,True,not_available,No direct indicator in this NFHS-5 CSV.
3,recent_separation,She left him in the past year,2.32,2,original_DA,2,True,not_available,No direct indicator in this NFHS-5 CSV.
4,lethal_weapon,Used/threatened with a lethal weapon,2.64,2,original_DA,2,True,not_available,No direct indicator in this NFHS-5 CSV.
5,avoids_arrest,Lied about behavior/avoided being arrested,3.52,3,original_DA,3,True,not_available,No direct indicator in this NFHS-5 CSV.
6,strangulation,Choking/Strangulation,2.05,3,original_DA,2,False,not_available,No direct indicator in this NFHS-5 CSV.
7,illegal_drug_use,Partner uses illegal drugs,1.86,2,original_DA,2,True,not_available,No direct partner drug-use indicator in this a...
8,problem_drinking,He is an alcoholic/problem drinker,1.87,2,original_DA,2,True,context_only,"Alcohol prevalence is available, but partner p..."
9,violence_during_pregnancy,She was beaten while pregnant,5.60,4,original_DA,4,True,direct_reference,Physical violence during any pregnancy is dire...


In [14]:
mapping=pd.DataFrame(mapping_rows,columns=['feature','nfhs_status','mapping_note'])
dawi_mapping=dawi.merge(mapping,on='feature',how='left')
display(dawi_mapping)

,feature,dawi_item,rrr,weight,source_group,rule_weight,weight_matches_rule,nfhs_status,mapping_note
0,violence_escalation,Increase in severity/frequency of physical vio...,1.90,2,original_DA,2,True,related,Spousal violence is available; escalation itse...
1,threat_to_kill,He threatened to kill,1.76,1,original_DA,1,True,not_available,No direct indicator in this NFHS-5 CSV.
2,violent_jealousy,He is constantly/violently jealous,3.32,3,original_DA,3,True,not_available,No direct indicator in this NFHS-5 CSV.
3,recent_separation,She left him in the past year,2.32,2,original_DA,2,True,not_available,No direct indicator in this NFHS-5 CSV.
4,lethal_weapon,Used/threatened with a lethal weapon,2.64,2,original_DA,2,True,not_available,No direct indicator in this NFHS-5 CSV.
5,avoids_arrest,Lied about behavior/avoided being arrested,3.52,3,original_DA,3,True,not_available,No direct indicator in this NFHS-5 CSV.
6,strangulation,Choking/Strangulation,2.05,3,original_DA,2,False,not_available,No direct indicator in this NFHS-5 CSV.
7,illegal_drug_use,Partner uses illegal drugs,1.86,2,original_DA,2,True,not_available,No direct partner drug-use indicator in this a...
8,problem_drinking,He is an alcoholic/problem drinker,1.87,2,original_DA,2,True,context_only,"Alcohol prevalence is available, but partner p..."
9,violence_during_pregnancy,She was beaten while pregnant,5.60,4,original_DA,4,True,direct_reference,Physical violence during any pregnancy is dire...


In [15]:
spousal=nfhs_violence['sub indicators'].astype(str).str.contains('spousal violence',case=False,na=False)
pregnancy=nfhs_violence['sub indicators'].astype(str).str.contains('physical violence during any pregnancy',case=False,na=False)
sexual=nfhs_violence['sub indicators'].astype(str).str.contains('sexual violence',case=False,na=False)
print('Spousal violence state rows:',spousal.sum())
print('Pregnancy violence state rows:',pregnancy.sum())
print('Sexual violence state rows:',sexual.sum())
display(nfhs_violence.loc[spousal|pregnancy|sexual].head(20))

Spousal violence state rows: 37
Pregnancy violence state rows: 37
Sexual violence state rows: 37


,Indicators,sub indicators,NFHS-5 (2019-20),STATE/UT
0,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,23.2,Andaman & Nicobar Islands
1,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,(0.0),Andaman & Nicobar Islands
2,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced se...,1.4,Andaman & Nicobar Islands
3,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,28.8,Andhra Pradesh
4,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,3.5,Andhra Pradesh
5,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced se...,3.8,Andhra Pradesh
6,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,26.6,Assam
7,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,2.2,Assam
8,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced se...,7.4,Assam
9,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,40.6,Bihar


In [16]:
# 3. DA-WI weighted reference score
# This reproduces the published weighted DA-WI structure. It is a reference score, not the final target label for our app model.

In [17]:
DAWI_WEIGHTS=dict(zip(dawi['feature'],dawi['weight']))
def calculate_dawi_score(row):
    return sum(int(row.get(feature,0))*weight for feature,weight in DAWI_WEIGHTS.items())
print('Number of factors:',len(DAWI_WEIGHTS))
print('Maximum weight sum from this table:',sum(DAWI_WEIGHTS.values()))
print('Published DA-WI weighted score range: 0–64')

Number of factors: 26
Maximum weight sum from this table: 64
Published DA-WI weighted score range: 0–64


In [18]:

## 4. Save outputs for Notebook 03


In [19]:
os.makedirs('../data/processed',exist_ok=True)
dawi.to_csv('../data/processed/dawi_risk_factors.csv',index=False)
dawi_mapping.to_csv('../data/processed/dawi_nfhs_mapping.csv',index=False)
print('Saved dawi_risk_factors.csv and dawi_nfhs_mapping.csv')

Saved dawi_risk_factors.csv and dawi_nfhs_mapping.csv


In [20]:
# Frozen output
# Notebook 03 will use the 26 DA-WI factors, published RRRs/weights, and the NFHS prevalence references. App-specific immediate-safety variables remain separate from DA-WI factors.

In [21]:
print("DA-WI factors:", len(dawi))
print("Total published weight:", dawi["weight"].sum())
print("Unexpected mismatches:", len(unexpected_mismatches))

DA-WI factors: 26
Total published weight: 64
Unexpected mismatches: 0


In [22]:
print("NFHS violence reference:", nfhs_violence.shape)

display(
    nfhs_violence[
        ["Indicators", "sub indicators", "NFHS-5 (2019-20)", "STATE/UT"]
    ].head(15)
)

NFHS violence reference: (111, 4)


,Indicators,sub indicators,NFHS-5 (2019-20),STATE/UT
0,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,23.2,Andaman & Nicobar Islands
1,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,(0.0),Andaman & Nicobar Islands
2,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced se...,1.4,Andaman & Nicobar Islands
3,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,28.8,Andhra Pradesh
4,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,3.5,Andhra Pradesh
5,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced se...,3.8,Andhra Pradesh
6,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,26.6,Assam
7,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,2.2,Assam
8,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced se...,7.4,Assam
9,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,40.6,Bihar


In [23]:
import os

print(
    os.path.exists("../data/processed/dawi_risk_factors.csv"),
    os.path.exists("../data/processed/dawi_nfhs_mapping.csv")
)

print(
    "DA-WI factors:",
    pd.read_csv("../data/processed/dawi_risk_factors.csv").shape
)

print(
    "DA-WI mapping:",
    pd.read_csv("../data/processed/dawi_nfhs_mapping.csv").shape
)

True True
DA-WI factors: (26, 7)
DA-WI mapping: (26, 9)
